# ATML PA2 - Kaggle launcher (Task 4, safety calibration)

Thin launcher only. All experiment logic lives in the `.py` files.

Notebook settings before running:

- Accelerator: **GPU T4 x2**.
- Internet: **On**.
- Add Input: the dataset made from `task4_adapters.zip`, which holds the three final standard adapters under `outputs/task1_dpo/standard`, `outputs/task2_ppo/standard` and `outputs/task3_grpo/standard`.

Generation and judging both skip what is already saved, so every cell is safe to rerun. Run section 11 and download the zip after each long step.

### 1. GPU check

In [ ]:
!nvidia-smi

In [ ]:
%env CUDA_VISIBLE_DEVICES=0
%env TOKENIZERS_PARALLELISM=false

### 2. Clone or update the repo

In [ ]:
%cd /kaggle/working
!test -d atml_PA2 || git clone https://github.com/Zeeba1705/atml_PA2.git
%cd /kaggle/working/atml_PA2
!git fetch origin
!git checkout task4-safety
!git pull
!git log --oneline -3

### 3. Install the pinned requirements

In [ ]:
!pip install -q -r requirements.txt

### 4. Course assets

Downloads only when validation fails.

In [ ]:
!python -m scripts.validate_assets || (python -m scripts.download_assets && python -m scripts.validate_assets)

### 5. Copy in the adapters (and any earlier Task 4 results)

Copies every `outputs/` and `results/` folder found in the attached datasets into the repo, without overwriting newer files. The checksums printed at the end identify the three adapters.

In [ ]:
%%bash
cd /kaggle/working/atml_PA2
mkdir -p outputs results

echo "attached inputs:"
find /kaggle/input -maxdepth 5 | head -40

#an upload that Kaggle left zipped is unpacked into the repo
for z in $(find /kaggle/input -name "*.zip"); do
    unzip -n -q "$z" -d . && echo "unzipped $z"
done

#an upload that Kaggle unpacked: every outputs/ or results/ folder found at any depth is copied in
for d in $(find /kaggle/input -type d \( -name outputs -o -name results \)); do
    cp -rn "$d/." "$(basename $d)/" && echo "restored $(basename $d) from $d"
done

md5sum outputs/task1_dpo/standard/adapter_model.safetensors outputs/task2_ppo/standard/adapter_model.safetensors outputs/task3_grpo/standard/adapter_model.safetensors

### 6. Environment check and unit tests

In [ ]:
!python -m scripts.check_environment
!python -m pytest tests -q

### 7. Quick test

4 safe and 4 unsafe prompts through all four policies, the judge and the evaluation, written to `results/task4_safety_quicktest/`. Run this before the full run.

In [ ]:
!python -m task4_safety.generate_responses --config configs/feedback.yaml --examples-per-class 4
!python -m task4_safety.judge_responses --config configs/feedback.yaml --examples-per-class 4
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml --examples-per-class 4

### 8. Generate: one greedy response per policy for all 450 XSTest prompts

Finished policies are skipped.

In [ ]:
!python -m task4_safety.generate_responses --config configs/feedback.yaml

### 9. Judge every response with the supplied judge

One call per response, 1800 in total. Labels are saved one by one, so after a dead session this cell continues where it stopped.

In [ ]:
!python -m task4_safety.judge_responses --config configs/feedback.yaml

### 10. Audit sheet and safety tables

`make_audit_sheet` writes `results/task4_safety/manual_audit_sheet.csv` to label by hand. `evaluate_safety` prints the aggregate rates only; the agreement with the manual labels is computed later on the laptop with `--audit-done`.

In [ ]:
!python -m task4_safety.make_audit_sheet --config configs/feedback.yaml
!python -m task4_safety.evaluate_safety --config configs/feedback.yaml

### 11. Pack the results for download

Writes `/kaggle/working/atml_pa2_task4.zip`. Download it from the Output panel.

In [ ]:
%%bash
cd /kaggle/working/atml_PA2
rm -f /kaggle/working/atml_pa2_task4.zip
zip -qr /kaggle/working/atml_pa2_task4.zip results/task4_safety results/task4_safety_quicktest
ls -la /kaggle/working/atml_pa2_task4.zip